In [1]:
import requests
import json
import pandas as pd
import datetime


# -------------------------------------------
# CONFIGURATION
# -------------------------------------------
API_KEY = "Your API Key"
MODEL = "openai/gpt-4.1"
ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"


# -------------------------------------------
# PROMPT TEMPLATE (index-aligned, no markdown)
# -------------------------------------------
PROMPT_TEMPLATE = """
SYSTEM:
You are a company name standardization system.

TASK:
For each input company name, produce exactly one standardized company name.

STRICT RULES (NO EXCEPTIONS):
1. Output MUST be valid JSON only.
2. Do NOT use markdown, code blocks, or ```json.
3. Output length MUST equal input length.
4. Each input at index i MUST produce exactly one output at index i.
5. Do NOT skip, merge, reorder, or remove items.
6. It is allowed and expected to repeat standardized names.
7. Before responding, verify output count equals input count.

INPUT COMPANY NAMES (INDEXED):
{indexed_inputs}

REQUIRED OUTPUT FORMAT (JSON ONLY):

{{
  "standardized_names": [
    "<output for index 0>",
    "<output for index 1>",
    "<output for index 2>"
  ]
}}
"""

In [2]:

def format_indexed_inputs(name_list):
    return json.dumps(
        [f"{i}: {name}" for i, name in enumerate(name_list)],
        ensure_ascii=False
    )


def clean_llm_output(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```", 1)[-1]
        if text.lstrip().startswith("json"):
            text = text.lstrip()[4:]
        text = text.strip("`\n")
    return text.strip()


def standardize_names(name_list, max_retries=5):
    n = len(name_list)

    prompt = PROMPT_TEMPLATE.format(
        indexed_inputs=format_indexed_inputs(name_list)
    )

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
        "HTTP-Referer": "http://localhost",
        "X-Title": "Name Standardizer",
    }

    body = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "top_p": 0.3
    }

    last_error = None

    for _ in range(max_retries):
        response = requests.post(
            ENDPOINT, headers=headers, json=body, timeout=120
        )
        response.raise_for_status()

        raw = response.json()["choices"][0]["message"]["content"]
        cleaned = clean_llm_output(raw)

        try:
            data = json.loads(cleaned)
            names = data["standardized_names"]

            if not isinstance(names, list):
                raise ValueError("standardized_names is not a list")

            if len(names) != n:
                raise ValueError(f"Expected {n}, got {len(names)}")

            return names

        except Exception as e:
            last_error = e

    raise RuntimeError(f"Failed after retries: {last_error}")

In [3]:
def evaluate_model(
    data,
    batch_size=20,
    runs=10,
    num_ensemble=5,
):
    for k in range(1, runs + 1):
        print(f"Run {k}: {datetime.datetime.now()}")

        predictions = []

        for i in range(0, len(data), batch_size):
            batch = data.iloc[i:i+batch_size]["variant"].tolist()
            ensemble_pred = []
            for _ in range(num_ensemble):
                preds = standardize_names(batch)
                ensemble_pred.append(preds)
            # Simple majority vote
            predictions.extend([max(set(row), key=row.count) for row in zip(*ensemble_pred)])

        data_run = data.copy()
        data_run["prediction"] = predictions
        data_run["exact_match"] = (
            data_run["prediction"] == data_run["canonical"]
        ).astype(int)

        summary = (
            data_run
            .groupby("type")["exact_match"]
            .sum()
            .to_frame(name="correct")
        )

        summary.to_csv(
            f"ensemble_prompting_run_{k}_ensemble_{num_ensemble}.csv"
        )

In [4]:
data = pd.read_csv("selected_data2.csv")
print(data.groupby("type").size())

type
abbreviation    124
brand_parent    196
legal_suffix     57
partial         303
punctuation     232
typography      588
dtype: int64


In [10]:
evaluate_model(data, batch_size=200, num_ensemble=3)
# evaluate_model(data, batch_size=200, num_ensemble=5)

Run 1: 2026-04-05 16:01:14.136817
Run 2: 2026-04-05 16:04:44.111265
Run 3: 2026-04-05 16:08:25.256645
Run 4: 2026-04-05 16:11:57.151088
Run 5: 2026-04-05 16:15:27.945706
Run 6: 2026-04-05 16:19:16.551384
Run 7: 2026-04-05 16:22:31.184348
Run 8: 2026-04-05 16:25:53.872207
Run 9: 2026-04-05 16:29:24.418661
Run 10: 2026-04-05 16:32:54.222819


KeyboardInterrupt: 